In [1]:
# Install core vLLM and RAG dependencies optimized for ROCm
!pip install vllm --extra-index-url https://wheels.vllm.ai/rocm
!pip install chromadb sentence-transformers huggingface_hub

# Install dynamic document parsers
!pip install pypdf python-docx
print("✅ Core dependencies and dynamic parsers successfully installed.")

Looking in indexes: https://pypi.org/simple, https://wheels.vllm.ai/rocm

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 114.8 kB/s  0:02:43m0:00:0300:07
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 164.7 kB/s  0:00:03eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 166.8 kB/s  0:00:40 eta 0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 189.2 kB/s  0:00:24 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 280.3 kB/s  0:00:07 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 469.2 kB/s  0:00:48m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 278.6 kB/s  0:00:33m0:00:0100:02
  Attempting uninstall: aiohttp╺━━━━━━━━━━━ 19/29 [opentelemetry-semantic-conventions]
    Found existing installation: aiohttp 3.13.0━━━━━━━━━ 19/29 [opentelemetry-semantic-conventio

In [2]:
import os
import chromadb
from pypdf import PdfReader
from docx import Document
from sentence_transformers import SentenceTransformer

# 1. Dynamic File Extraction Engine
def extract_text_from_file(file_path):
    ext = os.path.splitext(file_path)[-1].lower()
    text = ""
    
    if ext == ".txt":
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()
    elif ext == ".pdf":
        reader = PdfReader(file_path)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    elif ext == ".docx":
        doc = Document(file_path)
        for paragraph in doc.paragraphs:
            if paragraph.text:
                text += paragraph.text + "\n"
    else:
        raise ValueError(f"Unsupported file format: {ext}")
        
    return text.strip()

# 2. Vector DB Infrastructure
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="compliance_rules")
embedder = SentenceTransformer('BAAI/bge-large-en-v1.5')

# Core Compliance Guidelines
compliance_rules = [
    "Rule 101: All individual investments or premium payments exceeding ₹50,000 mandate a verified Permanent Account Number (PAN) on record.",
    "Rule 102: Medical insurance policies must explicitly detail a 24-month waiting period for all pre-existing conditions.",
    "Rule 103: Corporate financial disclosures must include a clear 'Risk Factors' section outlining market volatility impacts.",
    "Rule 104: Any performance bonus disbursement above ₹10,000 must be accompanied by a tax-deduction at source (TDS) certificate."
]

# Populate the Database
embeddings = embedder.encode(compliance_rules).tolist()
collection.add(
    documents=compliance_rules,
    embeddings=embeddings,
    ids=[f"rule_{i}" for i in range(len(compliance_rules))]
)

print("✅ Ingestion engine ready. Vector Database populated.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ Ingestion engine ready. Vector Database populated.


In [3]:
from vllm import LLM, SamplingParams

# Deploy Qwen 2.5 on AMD GPUs
llm = LLM(
    model="Qwen/Qwen2.5-14B-Instruct",
    tensor_parallel_size=1, 
    max_model_len=8192,
    trust_remote_code=True
)

# Zero temperature enforces strict adherence to the rules without creative hallucination
sampling_params = SamplingParams(temperature=0.0, max_tokens=1500)
print("✅ Qwen 2.5 Inference Core deployed on ROCm.")

INFO 06-16 18:18:49 [__init__.py:224] Automatically detected platform rocm.
INFO 06-16 18:18:50 [utils.py:239] non-default args: {'trust_remote_code': True, 'max_model_len': 8192, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-14B-Instruct'}


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

INFO 06-16 18:18:58 [model.py:653] Resolved architecture: Qwen2ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 06-16 18:18:58 [model.py:1714] Using max model len 8192


...

(EngineCore_DP0 pid=520) INFO 06-16 18:19:07 [weight_utils.py:419] Using model weights format ['*.safetensors']
(EngineCore_DP0 pid=520) INFO 06-16 18:19:41 [weight_utils.py:440] Time spent downloading weights for Qwen/Qwen2.5-14B-Instruct: 33.682057 seconds


Loading safetensors checkpoint shards:   0% Completed | 0/8 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  12% Completed | 1/8 [00:00<00:04,  1.56it/s]
Loading safetensors checkpoint shards:  25% Completed | 2/8 [00:02<00:06,  1.12s/it]
Loading safetensors checkpoint shards:  38% Completed | 3/8 [00:03<00:06,  1.31s/it]
Loading safetensors checkpoint shards:  50% Completed | 4/8 [00:05<00:05,  1.40s/it]
Loading safetensors checkpoint shards:  62% Completed | 5/8 [00:06<00:04,  1.43s/it]
Loading safetensors checkpoint shards:  75% Completed | 6/8 [00:08<00:02,  1.46s/it]
Loading safetensors checkpoint shards:  88% Completed | 7/8 [00:09<00:01,  1.48s/it]
Loading safetensors checkpoint shards: 100% Completed | 8/8 [00:11<00:00,  1.50s/it]
Loading safetensors checkpoint shards: 100% Completed | 8/8 [00:11<00:00,  1.40s/it]
(EngineCore_DP0 pid=520) 


(EngineCore_DP0 pid=520) INFO 06-16 18:19:53 [default_loader.py:314] Loading weights took 11.35 seconds
(EngineCore_DP0 pid=520) INFO 06-16 18:19:53 [gpu_model_runner.py:2912] Model loading took 27.6289 GiB and 45.950251 seconds
(EngineCore_DP0 pid=520) INFO 06-16 18:19:59 [backends.py:604] Using cache directory: /root/.cache/vllm/torch_compile_cache/08d5880bd2/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=520) INFO 06-16 18:19:59 [backends.py:618] Dynamo bytecode transform time: 5.50 s
(EngineCore_DP0 pid=520) WARNING 06-16 18:19:59 [partition_rules.py:43] Failed to resolve operator for Inductor partition: vllm::mamba_mixer2
(EngineCore_DP0 pid=520) WARNING 06-16 18:19:59 [partition_rules.py:43] Failed to resolve operator for Inductor partition: vllm::mamba_mixer
(EngineCore_DP0 pid=520) WARNING 06-16 18:19:59 [partition_rules.py:43] Failed to resolve operator for Inductor partition: vllm::short_conv
(EngineCore_DP0 pid=520) WARNING 06-16 18:19:59 [partition_rules.py:

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:04<00:00, 16.08it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 67/67 [00:09<00:00,  7.28it/s]


(EngineCore_DP0 pid=520) INFO 06-16 18:21:01 [gpu_model_runner.py:3802] Graph capturing finished in 14 secs, took 0.84 GiB
(EngineCore_DP0 pid=520) INFO 06-16 18:21:01 [core.py:240] init engine (profile, create kv cache, warmup model) took 68.50 seconds
(EngineCore_DP0 pid=520) INFO 06-16 18:21:02 [gc_utils.py:40] GC Debug Config. enabled:False,top_objects:-1
INFO 06-16 18:21:02 [llm.py:337] Supported tasks: ['generate']
✅ Qwen 2.5 Inference Core deployed on ROCm.


In [4]:
import json

def run_compliance_audit(file_path):
    # Step 1: Extract text dynamically
    try:
        document_content = extract_text_from_file(file_path)
        print(f"📖 Extracted text from: {file_path}")
    except Exception as e:
        return f"Extraction Error: {str(e)}"
    
    # Step 2: Retrieve relevant rules
    doc_embedding = embedder.encode([document_content]).tolist()
    query_results = collection.query(query_embeddings=doc_embedding, n_results=2)
    matched_rules = "\n".join(query_results['documents'][0])
    
    # Step 3: Strict Prompting for Auditable Output
    prompt = f"""<|im_start|>system
You are an expert financial and corporate compliance auditor. Cross-examine the provided document against the system rules. 
Output your decision strictly as a JSON object. Do not include markdown formatting like ```json or any trailing text.

Target Schema:
{{
  "audit_status": "PASS" or "FAIL",
  "confidence_score": 0.0 to 1.0,
  "executive_summary": "High-level summary of the audit findings",
  "violations": [
    {{
      "rule_id": "Rule ID number",
      "breach_description": "Explanation of how the rule was broken",
      "extracted_evidence": "The direct quote from the document proving the breach"
    }}
  ]
}}
<|im_end|>
<|im_start|>user
[Target Rules]
{matched_rules}

[Document Content]
{document_content}
<|im_end|>
<|im_start|>assistant
"""

    # Step 4: Generate Report
    outputs = llm.generate([prompt], sampling_params, use_tqdm=False)
    raw_response = outputs[0].outputs[0].text.strip()
    
    # Cleanup in case the model leaks markdown ticks
    if raw_response.startswith("```"):
        raw_response = raw_response.strip("```json").strip("```").strip()
        
    return raw_response

In [5]:
# Generate a test text document (You can swap this with a real .pdf or .docx path in your hackathon demo)
sample_file_path = "client_application_Yadavalli_Venkata_Shanmukh.txt"

with open(sample_file_path, "w", encoding="utf-8") as f:
    f.write("""
    Client Name: Yadavalli Venkata Shanmukh
    Age/Gender: 22 Years/ Male
    Transaction: Corporate Performance Bonus Disbursement
    Disbursement Value: ₹20,000
    Notes: Bonus processed successfully based on recent project deliverables. Review pending. No Tax Deduction at Source (TDS) certification has been attached or filed at this time.
    """)

# Execute the Audit
print("Executing AI Audit Pipeline...\n")
raw_report = run_compliance_audit(sample_file_path)

# Parse and format the output beautifully
try:
    parsed_report = json.loads(raw_report)
    print("\n" + "="*50)
    print(f"🚨 AUDIT STATUS: {parsed_report['audit_status']} 🚨")
    print("="*50)
    print(json.dumps(parsed_report, indent=2))
except Exception as e:
    print("Output parsing failed. Raw response:")
    print(raw_report)

Executing AI Audit Pipeline...

📖 Extracted text from: client_application_Yadavalli_Venkata_Shanmukh.txt

🚨 AUDIT STATUS: FAIL 🚨
{
  "audit_status": "FAIL",
  "confidence_score": 0.95,
  "executive_summary": "The document fails to comply with Rule 104 as the performance bonus disbursement of \u20b920,000 does not have a TDS certificate attached.",
  "violations": [
    {
      "rule_id": "Rule 104",
      "breach_description": "The performance bonus disbursement of \u20b920,000 exceeds \u20b910,000 and lacks a TDS certificate.",
      "extracted_evidence": "No Tax Deduction at Source (TDS) certification has been attached or filed at this time."
    }
  ]
}
